# TSFabrics pilot: Stage 1 (CPU only, no images), amendment v1.2.2

This notebook only **launches** `scripts/run_stage1.py`. All logic lives in the repository.

**Inputs to attach (Add Input):**
1. The TSFabrics dataset (the one used for the forensic audit; it contains `CSV/`, `IMAGE/`, `ANNOTATION/`).
2. This repository, uploaded as a Kaggle dataset (zip of the GitHub repo).

**Settings:** Accelerator **None** (CPU). **Internet ON** (pinned packages are installed into an isolated folder).

**What Stage 1 reads:** synthetic data, the frozen CSVs and the 22 per-scenario label CSVs. It opens **no image**, lists no
image file, and computes no detector output.

**Run:** Run All. Expected time: about 30 to 60 minutes (the slow tests and the CPU detector tests dominate).

**PASS:** the last cell prints `STAGE 1 RESULT: PASS` with `failed: 0, skipped: 0`.

**Send back:** the last cell's printout and the files in `/kaggle/working/stage1_out/`. They become
`audit/stage1_kaggle_v1_2_2/` in the repository.

In [ ]:
import os, sys, json, shutil, subprocess, hashlib

EXPECTED_PREREG_SHA256 = "182df443b29e8b5ee1922de61219871c1a6e052f538fd8ef1c4b4466db4a70dd"
SKIP_DIRS = {"IMAGE", "ANNOTATION", "CSV", ".git", "__pycache__"}

def bounded_walk(top, max_depth=5):
    """os.walk that never descends into dataset image/annotation/label folders."""
    base_depth = top.rstrip(os.sep).count(os.sep)
    for base, dirs, files in os.walk(top):
        dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS)
        if base.count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
        yield base, dirs, files

# 1. Locate the repository copy under /kaggle/input (must contain PREREGISTRATION.md and src/tsfpilot)
repos = [b for b, d, f in bounded_walk("/kaggle/input")
         if "PREREGISTRATION.md" in f and os.path.isdir(os.path.join(b, "src", "tsfpilot"))]
assert len(repos) == 1, f"expected exactly one repository copy under /kaggle/input, found: {repos}"
WORK = "/kaggle/working/repo"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(repos[0], WORK, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
prereg = hashlib.sha256(open(f"{WORK}/PREREGISTRATION.md", "rb").read()).hexdigest()
assert prereg == EXPECTED_PREREG_SHA256, f"PREREGISTRATION.md hash mismatch: {prereg}"
print("repository:", repos[0], "->", WORK, "| PREREGISTRATION.md hash OK")

In [ ]:
# 2. Install the pinned packages into an isolated folder (the Kaggle base environment is not modified).
# The suite now includes the Stage 2 CPU tests, so faiss is installed too; torch, torchvision and timm come from
# the Kaggle image. A skipped test makes Stage 1 FAIL, so every import is checked here.
DEPS = "/tmp/stage1_pydeps"
shutil.rmtree(DEPS, ignore_errors=True)
for args in (["-r", f"{WORK}/requirements.txt"], ["--no-deps", "-r", f"{WORK}/requirements-stage2.txt"]):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-warn-script-location",
                        "--target", DEPS, *args], capture_output=True, text=True)
    print(r.stdout[-1000:], r.stderr[-1500:])
    assert r.returncode == 0, "pip install failed: is Internet ON? (see the error above)"
ENV = dict(os.environ, PYTHONPATH=DEPS, PYTHONNOUSERSITE="1")
chk = subprocess.run([sys.executable, "-c",
    "import sys, numpy, scipy, pandas, numba, cv2, yaml, pytest, torch, torchvision, timm, faiss; "
    "torch.from_numpy(numpy.zeros(3)); "
    "print(sys.version.split()[0], numpy.__version__, scipy.__version__, pandas.__version__, numba.__version__, "
    "cv2.__version__, torch.__version__, timm.__version__, faiss.__version__)"],
    env=ENV, capture_output=True, text=True)
print("python numpy scipy pandas numba cv2 torch timm faiss:", chk.stdout.strip(), chk.stderr[-1500:])
assert chk.returncode == 0 and chk.stdout.split()[1] == "2.4.4", "pinned packages are not the ones imported"

In [ ]:
# 3. Locate the TSFabrics dataset root (the folder containing CSV/ and IMAGE/). Only folder names are listed.
roots = []
for base, dirs, files in os.walk("/kaggle/input"):
    if "CSV" in dirs and "IMAGE" in dirs:
        roots.append(base)
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
assert len(roots) == 1, f"expected exactly one TSFabrics root, found: {roots}"
ROOT = roots[0]
n_csv = len([f for f in os.listdir(os.path.join(ROOT, "CSV")) if f.endswith(".csv")])
print("dataset root:", ROOT, "| label CSV files:", n_csv)
assert n_csv == 22, "expected 22 per-scenario label CSVs"

In [ ]:
# 4. Run Stage 1 (pytest suite + report) in a fresh interpreter with the pinned packages. 30 to 60 minutes.
OUT = "/kaggle/working/stage1_out"
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/stage1_log.txt", "w") as log:
    r = subprocess.run([sys.executable, "scripts/run_stage1.py", "--dataset-root", ROOT], cwd=WORK, env=ENV,
                       stdout=log, stderr=subprocess.STDOUT, text=True)
print("exit code:", r.returncode)
print(open(f"{OUT}/stage1_log.txt").read()[-3000:])

In [ ]:
# 5. Summary and outputs to save
rep = json.load(open(f"{WORK}/audit/stage1_report.json"))
for f in ("audit/stage1_report.json", "audit/bank_frames_v1_2_1.csv", "audit/stage1_junit.xml"):
    if os.path.exists(f"{WORK}/{f}"):
        shutil.copy(f"{WORK}/{f}", OUT)
print("STAGE 1 RESULT:", rep["result"], "|", ", ".join(f"{k}: {v}" for k, v in rep["counts"].items()))
print("blocking failures:", rep["blocking_failures"] or "none")
print("C6 reportable:", rep["c6_reportable"], "| dataset root found:", rep["dataset_root_found"])
print("files fingerprint:", rep["files_fingerprint"])
print("config sha256:", rep["config_sha256"], "| config version:", rep.get("config_version"))
BANK_DL0012 = "534535eb8ea947bed4fb9c9a03352f52ba0ba197ba7694385b5735e9b936a535"
print("bank frames identical to the DL-0012 bank:", (rep.get("bank_frames_csv") or {}).get("sha256") == BANK_DL0012)
print("bank frames CSV:", rep.get("bank_frames_csv"))
e = rep["environment"]
print("python", e["python"].split()[0], "| numpy", e["numpy"], "| numba", e["numba"], "| cv2", e["cv2"])
for t in rep["tests"]:
    if t["status"] != "passed":
        print(t["status"].upper(), t["module"], "::", t["test"])
print("saved:", sorted(os.listdir(OUT)))